# Solution A2 · Gradient descent for linear regression

**Type**: Implement from scratch

## Objective

Implement gradient descent for OLS, show that on unscaled features it diverges, and compute the maximum stable
learning rate starting from the Hessian.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

rng = np.random.default_rng(0)
n = 300
area = rng.normal(90, 30, n)                     # square meters
income = rng.normal(30_000, 10_000, n)           # euros
X = np.column_stack([area, income])
y = 50 + 2.0 * area + 0.003 * income + rng.normal(0, 5, n)

A_raw = np.column_stack([np.ones(n), X])

## Commented solution

The MSE is `(1/n)·‖Aw − y‖²`: its gradient is `(2/n)·Aᵀ(Aw − y)` and its Hessian `H = (2/n)·AᵀA`, constant.
A quadratic function with a positive definite Hessian is convex, so gradient descent converges to the OLS
solution if the learning rate is small enough. "Small enough" has a precise answer: `lr < 2/λ_max(H)`.

In [ ]:
def gradient_descent(A, y, lr, n_iter):
    n = len(y)
    w = np.zeros(A.shape[1])
    history = []
    for _ in range(n_iter):
        residual = A @ w - y
        with np.errstate(all="ignore"):
            mse = np.mean(residual ** 2)
        history.append(mse)
        if not np.isfinite(mse) or mse > 1e30:
            break
        w = w - lr * (2 / n) * A.T @ residual
    return w, np.array(history)

In [ ]:
w_raw, history_raw = gradient_descent(A_raw, y, lr=0.01, n_iter=200)
diverges_raw = history_raw[-1] > history_raw[0]
print("initial MSE:", f"{history_raw[0]:.3g}", "| last recorded:", f"{history_raw[-1]:.3g}",
      "| iterations done:", len(history_raw))

In [ ]:
X_std = (X - X.mean(axis=0)) / X.std(axis=0)
A_std = np.column_stack([np.ones(n), X_std])
w_gd, history_std = gradient_descent(A_std, y, lr=0.1, n_iter=500)
w_ols = np.linalg.lstsq(A_std, y, rcond=None)[0]
print("w_gd :", w_gd.round(4))
print("w_ols:", w_ols.round(4))

In [ ]:
H = (2 / n) * A_std.T @ A_std
lr_max = 2 / np.linalg.eigvalsh(H).max()
print("lr_max (standardized data):", round(lr_max, 4))

w_a, history_a = gradient_descent(A_std, y, lr=0.9 * lr_max, n_iter=500)
w_b, history_b = gradient_descent(A_std, y, lr=1.1 * lr_max, n_iter=500)
conv_09 = history_a[-1] < history_a[0] and close(w_a, w_ols, atol=1e-3)
div_11 = history_b[-1] > history_b[0]
print("0.9 * lr_max converges:", conv_09, "| 1.1 * lr_max diverges:", div_11)

In [ ]:
H_raw = (2 / n) * A_raw.T @ A_raw
lr_max_raw = 2 / np.linalg.eigvalsh(H_raw).max()
print("lr_max (raw data):", f"{lr_max_raw:.3g}", "| ratio:", f"{lr_max / lr_max_raw:.3g}")

## Check

In [ ]:
with verify():
    check(isinstance(history_std, np.ndarray) and len(history_std) == 500,
          "the function returns the MSE history as an array", "`history` must be an array with one MSE per iteration")
    check(bool(diverges_raw), "on the raw data gradient descent diverges",
          "with lr=0.01 on unscaled data the MSE should explode")
    check(close(w_gd, w_ols, atol=1e-3), "on the standardized data you reach the OLS solution",
          "w_gd does not match the exact solution: check the gradient (2/n) A^T (A w - y)")
    check(np.all(np.diff(history_std) <= 1e-12), "the MSE never grows (convex function, small lr)",
          "with a small learning rate the MSE should go down at every step")
    lr_expected = 2 / np.linalg.eigvalsh((2 / n) * A_std.T @ A_std).max()
    check(close(lr_max, lr_expected, rtol=1e-6), "correct lr_max",
          "lr_max = 2 / (largest eigenvalue of H), with H = (2/n) A^T A")
    check(bool(conv_09) and bool(div_11), "0.9*lr_max converges and 1.1*lr_max diverges",
          "conv_09 must be True and div_11 True: compare the last MSE with the first")
    check(lr_max_raw < 1e-6 and lr_max / lr_max_raw > 1e6,
          "on the raw data the stable learning rate is millions of times smaller",
          "compute lr_max on the raw matrix: it should come out of the order of 1e-9")
done()

## What you learned

- The stability of gradient descent on a quadratic is governed by λ_max(H): `lr < 2/λ_max`. If you exceed it
  the error is multiplied by a factor greater than 1 at every step.
- Income (order 10⁴) makes λ_max grow by almost 9 orders of magnitude: to avoid diverging you need a learning rate
  of the order of 10⁻⁹, with which the other direction (the area) practically never moves. Standardizing
  brings all the eigenvalues to the same order of magnitude and lets you use large steps.
- Convexity means there is a single minimum; it does **not** mean that any learning rate reaches it.

## Common pitfall

At the first `nan` in the loss you tend to blame the data or the code. With gradient descent the most frequent cause is a
learning rate too large for the scale of the features. Before looking for bugs elsewhere, reduce lr by an order of
magnitude and scale the features.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why is gradient descent slower (or does it diverge) when the features have very different scales, even though OLS has an exact solution?</summary>

Because the surface of the MSE becomes a very long and narrow valley: the Hessian has very different eigenvalues. The
learning rate is limited by the largest eigenvalue (2/λ_max) but the speed along the direction with the smallest eigenvalue
is proportional to lr·λ_min, so the ratio λ_max/λ_min (the conditioning) decides how many iterations are needed.
Standardizing brings the eigenvalues closer to each other.

</details>